# Crohn's Disease scRNA-seq Drug Repurposing Pipeline

**Glen Ritschel | Ritschel Research | 2026**

GitHub: https://github.com/glenritschel/ibd-scrna

**Dataset**: GSE134809 (Smillie et al.) — 11 paired ileal Involved + 11 Uninvolved biopsies.
MTX format, gene symbols (CellRanger v2). PBMC samples excluded.

**Primary comparison**: Involved (inflamed) vs Uninvolved — paired disease-specific DE

**Strategy**: Drive is mounted first. All outputs save directly to Drive after each step.
If the runtime resets, rerun from the failed step — completed steps reload from Drive.

---
**Before running**: Runtime > Change runtime type > **T4 GPU**

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive mounted.")
print("Output directory:", DRIVE_DIR)
print("Existing files:", sorted(os.listdir(DRIVE_DIR)))

## Step 2 — Clone repo and install dependencies

In [ ]:
import os
if not os.path.exists("/content/ibd-scrna"):
    !git clone https://github.com/glenritschel/ibd-scrna.git
else:
    !cd /content/ibd-scrna && git pull
%cd /content/ibd-scrna/notebooks
print("Working directory:", os.getcwd())

In [ ]:
import subprocess, sys
packages = [
    "scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10",
    "leidenalg", "python-igraph", "scikit-misc", "biopython"
]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Runtime > Change runtime type > T4 GPU")

## Step 3 — Download raw data to Drive

Downloads GSE134809_RAW.tar (579 MB) to Drive.
Extracts flat MTX files (barcodes, genes, matrix per sample).
No Ensembl mapping needed — files already use gene symbols.

In [ ]:
import os, urllib.request, tarfile, glob

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
RAW_DIR   = os.path.join(DRIVE_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

TAR_URL  = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE134nnn/GSE134809/suppl/GSE134809_RAW.tar"
TAR_PATH = os.path.join(RAW_DIR, "GSE134809_RAW.tar")

if not os.path.exists(TAR_PATH):
    print("Downloading GSE134809_RAW.tar (579 MB) to Drive...", flush=True)
    urllib.request.urlretrieve(TAR_URL, TAR_PATH)
    print("Done.", round(os.path.getsize(TAR_PATH) / 1e6), "MB")
else:
    print("TAR already on Drive.")

if len(glob.glob(os.path.join(RAW_DIR, "GSM*_barcodes.tsv.gz"))) < 10:
    print("Extracting...")
    with tarfile.open(TAR_PATH) as tf:
        tf.extractall(RAW_DIR)
    print("Extraction complete.")

barcodes = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*_barcodes.tsv.gz")))
print(f"Sample sets ready: {len(barcodes)}")
# Show first few
for f in barcodes[:4]:
    print(" ", os.path.basename(f))

## Step 4 — Load, QC, and save to Drive

Loads 22 ileal samples (11 Involved + 11 Uninvolved). PBMC samples are excluded.
**If this step completed in a previous run, it reloads from Drive and skips loading.**

In [ ]:
import os, gc, shutil, glob, re, tempfile
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
RAW_DIR   = os.path.join(DRIVE_DIR, "raw")
QC_PATH   = os.path.join(DRIVE_DIR, "adata_qc.h5ad")

QC_MIN_GENES   = 200
QC_MIN_CELLS   = 3
QC_MAX_MITO    = 20.0
N_HVG          = 4000
USE_CONDITIONS = ["Involved", "Uninvolved"]

SAMPLE_MAP = {
    "GSM3972009": ("Involved",   "69"),
    "GSM3972010": ("Uninvolved", "68"),
    "GSM3972011": ("Involved",   "122"),
    "GSM3972012": ("Uninvolved", "123"),
    "GSM3972013": ("Involved",   "128"),
    "GSM3972014": ("Uninvolved", "129"),
    "GSM3972015": ("Uninvolved", "135"),
    "GSM3972016": ("Involved",   "138"),
    "GSM3972017": ("Involved",   "158"),
    "GSM3972018": ("Uninvolved", "159"),
    "GSM3972019": ("Uninvolved", "180"),
    "GSM3972020": ("Involved",   "181"),
    "GSM3972021": ("Uninvolved", "186"),
    "GSM3972022": ("Involved",   "187"),
    "GSM3972023": ("Uninvolved", "189"),
    "GSM3972024": ("Involved",   "190"),
    "GSM3972025": ("Uninvolved", "192"),
    "GSM3972026": ("Involved",   "193"),
    "GSM3972027": ("Uninvolved", "195"),
    "GSM3972028": ("Involved",   "196"),
    "GSM3972029": ("Uninvolved", "208"),
    "GSM3972030": ("Involved",   "209"),
}

def load_mtx_flat(raw_dir, gsm, condition, patient):
    prefix = gsm + "_" + patient + "_"
    barcodes_path = os.path.join(raw_dir, prefix + "barcodes.tsv.gz")
    genes_path    = os.path.join(raw_dir, prefix + "genes.tsv.gz")
    matrix_path   = os.path.join(raw_dir, prefix + "matrix.mtx.gz")
    for p in [barcodes_path, genes_path, matrix_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing: {p}")
    tmpdir = tempfile.mkdtemp()
    try:
        shutil.copy(barcodes_path, os.path.join(tmpdir, "barcodes.tsv.gz"))
        shutil.copy(genes_path,    os.path.join(tmpdir, "genes.tsv.gz"))
        shutil.copy(matrix_path,   os.path.join(tmpdir, "matrix.mtx.gz"))
        adata = sc.read_10x_mtx(tmpdir, var_names="gene_symbols", cache=False)
    finally:
        shutil.rmtree(tmpdir)
    adata.var_names_make_unique()
    sc.pp.filter_cells(adata, min_counts=1)
    sample_id = gsm + "_" + condition + "_" + patient
    adata.obs_names = [sample_id + "_" + bc for bc in adata.obs_names]
    adata.obs["sample"]    = sample_id
    adata.obs["condition"] = condition
    adata.obs["patient"]   = patient
    return adata

if os.path.exists(QC_PATH):
    print("QC object already on Drive. Loading...")
    adata_qc = sc.read_h5ad(QC_PATH)
    print("Loaded:", adata_qc.n_obs, "cells x", adata_qc.n_vars, "genes")
    for c in USE_CONDITIONS:
        print(f"  {c}:", (adata_qc.obs["condition"] == c).sum())
else:
    # Load by condition batch to manage memory
    batch_paths = []
    for cond in USE_CONDITIONS:
        batch_path = os.path.join(DRIVE_DIR, f"batch_{cond}.h5ad")
        if os.path.exists(batch_path):
            print(f"Batch {cond} already on Drive, skipping.")
            batch_paths.append(batch_path)
            continue
        print(f"\nLoading {cond} samples...")
        adatas = []
        for gsm, (condition, patient) in SAMPLE_MAP.items():
            if condition != cond: continue
            sid = gsm + "_" + cond + "_" + patient
            print(f"  {sid}...", end=" ", flush=True)
            try:
                a = load_mtx_flat(RAW_DIR, gsm, cond, patient)
                adatas.append(a)
                print(a.n_obs, "cells")
            except Exception as e:
                print("ERROR:", e)
        print(f"  Concatenating {cond}...")
        batch = sc.concat(adatas, join="outer", fill_value=0)
        del adatas; gc.collect()
        print(f"  {cond}: {batch.n_obs} cells — saving to Drive...")
        batch.write_h5ad(batch_path)
        del batch; gc.collect()
        print(f"  Saved: {batch_path}")
        batch_paths.append(batch_path)

    print("\nConcatenating batches from Drive...")
    adatas = [sc.read_h5ad(p) for p in batch_paths]
    adata  = sc.concat(adatas, join="outer", fill_value=0)
    del adatas; gc.collect()
    adata.layers["counts"] = sp.csr_matrix(adata.X) if not sp.issparse(adata.X) else adata.X.copy()
    print("Combined:", adata.n_obs, "cells x", adata.n_vars, "genes")

    print("\nRunning QC...")
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=QC_MIN_GENES)
    sc.pp.filter_genes(adata, min_cells=QC_MIN_CELLS)
    adata = adata[adata.obs["pct_counts_mt"] < QC_MAX_MITO].copy()
    print(f"QC: {n_before} -> {adata.n_obs} cells")

    print("Selecting HVGs...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["norm_log"] = adata.X.copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3",
                                layer="counts", batch_key="sample", subset=False)
    print("HVGs:", adata.var["highly_variable"].sum())

    print("Saving QC object to Drive...")
    adata.write_h5ad(QC_PATH)
    adata_qc = adata
    print("Saved:", QC_PATH)

print("\nStep 4 complete.")

## Step 5 — scVI Embedding + Leiden Clustering

In [ ]:
import os, gc
import numpy as np
import scanpy as sc
import scvi

DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
QC_PATH   = os.path.join(DRIVE_DIR, "adata_qc.h5ad")
SCVI_PATH = os.path.join(DRIVE_DIR, "adata_scvi.h5ad")
LEIDEN_RESOLUTIONS = [0.5, 0.8, 1.2]
RANDOM_SEED = 0

if os.path.exists(SCVI_PATH):
    print("scVI object already on Drive. Loading...")
    adata_scvi = sc.read_h5ad(SCVI_PATH)
    print("Loaded:", adata_scvi.n_obs, "cells,", adata_scvi.obs["leiden"].nunique(), "clusters")
else:
    if "adata_qc" not in dir():
        print("Loading QC object from Drive...")
        adata_qc = sc.read_h5ad(QC_PATH)
    adata_hvg = adata_qc[:, adata_qc.var["highly_variable"]].copy()
    print("HVG subset:", adata_hvg.n_obs, "x", adata_hvg.n_vars)

    import torch, random
    np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
    scvi.settings.seed = RANDOM_SEED
    scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
    model = scvi.model.SCVI(adata_hvg, n_latent=30, n_layers=2, n_hidden=128)
    print("Training scVI on", adata_hvg.n_obs, "cells...")
    model.train(max_epochs=400, early_stopping=False, accelerator="auto")
    final_loss = float(np.array(model.history["train_loss_epoch"].values[-1]).flat[0])
    print("Training complete. Final loss:", round(final_loss, 2))
    adata_hvg.obsm["X_scVI"] = model.get_latent_representation()

    sc.pp.neighbors(adata_hvg, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata_hvg)
    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_{res}"
        sc.tl.leiden(adata_hvg, resolution=res, key_added=key,
                     random_state=RANDOM_SEED, flavor="igraph",
                     n_iterations=2, directed=False)
        print(f"  Resolution {res}: {adata_hvg.obs[key].nunique()} clusters")

    adata_hvg.obs["leiden"] = adata_hvg.obs["leiden_0.8"].copy()
    print("Using resolution 0.8:", adata_hvg.obs["leiden"].nunique(), "clusters")
    print("Saving scVI object to Drive...")
    adata_hvg.write_h5ad(SCVI_PATH)
    adata_scvi = adata_hvg
    print("Saved:", SCVI_PATH)

print("\nStep 5 complete.")

## Step 6 — Copy scVI object locally for scripts 03–07

In [ ]:
import os, shutil
DRIVE_DIR       = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
LOCAL_PROCESSED = "/content/ibd-scrna/data/processed"
os.makedirs(LOCAL_PROCESSED, exist_ok=True)

for fname in ["adata_scvi.h5ad", "adata_qc.h5ad"]:
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join(LOCAL_PROCESSED, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        print(f"Copying {fname}...", end=" ", flush=True)
        shutil.copy2(src, dst)
        print(f"{os.path.getsize(dst)//1e6:.0f} MB")
    elif os.path.exists(dst):
        print(f"Already local: {fname}")
    else:
        print(f"NOT FOUND on Drive: {fname}")
print("Local processed:", os.listdir(LOCAL_PROCESSED))

## Step 7 — Script 03: Cell Type Annotation

In [ ]:
import os; os.chdir("/content/ibd-scrna/notebooks")
%run ../src/03_annotate_clusters.py

In [ ]:
import shutil, os
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
for fname in ["adata_annotated.h5ad", "cluster_annotations.csv", "cluster_marker_scores.csv"]:
    src = f"/content/ibd-scrna/data/processed/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, fname))
        print("Saved:", fname)

## Step 8 — Script 04: CD Signature Scoring

In [ ]:
import os; os.chdir("/content/ibd-scrna/notebooks")
%run ../src/04_signature_scoring.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
for f in glob.glob("/content/ibd-scrna/data/processed/signature_scores*.csv") + \
         ["/content/ibd-scrna/data/processed/adata_scored.h5ad"]:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
        print("Saved:", os.path.basename(f))

## Step 9 — Script 05: Differential Expression (Involved vs Uninvolved)

In [ ]:
import os; os.chdir("/content/ibd-scrna/notebooks")
%run ../src/05_differential_expression.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
for f in glob.glob("/content/ibd-scrna/data/processed/de_*.csv") + \
         ["/content/ibd-scrna/data/processed/adata_de.h5ad"]:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
        print("Saved:", os.path.basename(f))

## Step 10 — Script 06: LINCS L1000 Reversal Scoring

Takes 30–60 minutes. Do not close the browser tab.

In [ ]:
import os; os.chdir("/content/ibd-scrna/notebooks")
%run ../src/06_lincs_repurposing.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
for f in glob.glob("/content/ibd-scrna/data/processed/lincs_*.csv"):
    shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
    print("Saved:", os.path.basename(f))

## Step 11 — Script 07: Novelty Assessment + Priority Scoring

In [ ]:
import os; os.chdir("/content/ibd-scrna/notebooks")
%run ../src/07_novelty_prioritization.py

In [ ]:
import shutil, os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
for f in glob.glob("/content/ibd-scrna/data/processed/*.csv"):
    shutil.copy2(f, os.path.join(DRIVE_DIR, os.path.basename(f)))
    print("Saved:", os.path.basename(f))
print("\nPipeline complete. All outputs on Drive.")

## Step 12 — Verify outputs on Drive

In [ ]:
import os, glob
DRIVE_DIR = "/content/drive/MyDrive/Ritschel_Research/ibd_scrna_output"
files = sorted(
    glob.glob(os.path.join(DRIVE_DIR, "*.csv")) +
    glob.glob(os.path.join(DRIVE_DIR, "*.h5ad"))
)
print(f"{len(files)} output files on Drive:")
for f in files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f} MB)")